[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lbutler2405/EMP5027-rows-to-pixels/blob/main/notebooks/practical-5-geospatial-fundamentals/EMP5027-Lecture-5-Geospatial-Data-for-Environmental-Science.ipynb)


## Running this in Google Colab

Click the badge above to open this notebook directly in Colab, no local setup required. This notebook needs a couple of packages that are not preinstalled on Colab, and the cell below installs them automatically.

World and cities boundary data are fetched live from Natural Earth over HTTP, so we don't need to bundle any data files ourselves, just the geospatial packages.


In [ ]:
# --- Google Colab setup (safe to run locally too, it just skips this step) ---
import sys

if "google.colab" in sys.modules:
    !pip install -q geopandas rasterio rioxarray xarray

    print("Running in Colab, ready to go.")
else:
    print("Not running in Colab, assuming packages are already installed locally.")


In [ ]:
# Core imports: geopandas for vector data (points, lines, polygons with attributes),
# rasterio and rioxarray for raster data (gridded pixel values), xarray for the
# labelled array structures raster data end up in
import geopandas as gpd
import rasterio
import rioxarray as rxr
import xarray as xr
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Natural Earth hosts these boundary datasets as zipped shapefiles on S3.
# geopandas can read a zip straight off HTTP, no need to download and unpack it ourselves.
world = gpd.read_file(
    "https://naturalearth.s3.amazonaws.com/110m_cultural/ne_110m_admin_0_countries.zip"
)
cities = gpd.read_file(
    "https://naturalearth.s3.amazonaws.com/110m_cultural/ne_110m_populated_places.zip"
)


In [ ]:
import os
os.getcwd()


In [ ]:
world.to_csv("world_data.csv")


In [ ]:
cities.to_csv("cities_data.csv")


In [ ]:
# Extract Cyprus polygon
cyprus = world[world["ADMIN"] == "Cyprus"].copy()
cyprus


In [ ]:
# Reproject to a metric CRS (EPSG:3857, the Web Mercator projection used by most web maps)
cyprus_m = cyprus.to_crs(epsg=3857)

# Plot Cyprus (note: this plots the original geographic-CRS "cyprus", not "cyprus_m")
cyprus.plot(color="lightblue", edgecolor="k", figsize=(5, 5))
plt.title("Cyprus in EPSG:3857 (Meters)")
plt.xlabel("X (metres)")
plt.ylabel("Y (metres)")
plt.axis("equal")  # Keep proportions correct
plt.show()



In [ ]:
# Choose raster resolution in metres (smaller = finer grid, larger = faster).
res = 250  # a 250 m cell means each pixel covers a 250 m x 250 m area on the ground
# Get the bounding box (min/max X/Y) of the country polygon, in metres

minx, miny, maxx, maxy = iceland_m.total_bounds
# Compute raster size from the bounding box and chosen resolution

width  = int(np.ceil((maxx - minx) / res))   # number of columns (east-west)

height = int(np.ceil((maxy - miny) / res))   # number of rows (north-south)

from rasterio.transform import from_origin
# An affine transform maps array row/column indices to real-world coordinates.
# from_origin needs the top-left corner (minx, maxy) plus the pixel size in each direction.
transform = from_origin(minx, maxy, res, res)



In [ ]:
minx, miny, maxx, maxy


In [ ]:
width, height


In [ ]:
transform


In [ ]:
# Create 1D index arrays for the grid.
# Columns run 0 to width-1 (going east), rows run 0 to height-1 (going south).
cols = np.arange(width)     # column indices (west to east)
rows = np.arange(height)    # row indices (north to south)



In [ ]:
# Compute X and Y coordinates at each cell centre.
# We add/subtract 0.5 x resolution to land on the centre of each cell rather than its corner.
# In projected CRSs like this one, X increases eastward and Y decreases going south.
xs = minx + (cols + 0.5) * res  # X coordinates, eastwards from minx
ys = maxy - (rows + 0.5) * res  # Y coordinates, southwards from maxy

# Broadcast the 1D coordinate arrays into full 2D grids, one X and one Y value per cell
xx, yy = np.meshgrid(xs, ys)
xx, yy



## Raster layer A: distance-to-coast

We now have a grid, defined by `width`, `height` and `transform`, but no values in it yet. This section fills that grid with our first raster layer: for every cell, the straight-line distance to the coastline of the country polygon.

This is a genuinely common task in environmental science. Species distribution models, erosion risk models and a lot of coastal ecology work all use distance-to-coast, or distance to some other feature, as a predictor variable, because proximity to the coast correlates with salinity, wind exposure, temperature moderation and much else. The catch is that our environmental data are naturally a raster, a regular grid of values, while our coastline is naturally a vector, a set of connected line segments describing where land meets sea. Getting from one to the other, vector geometry to raster grid, is one of the most common operations in this field, so it is worth working through step by step rather than treating it as a black box.

The recipe we will follow, here and again for the land/sea mask and the synthetic sea surface temperature layer further down, is:

1. Get the geometry we care about as a shapely object, here the coastline as a single line.
2. Turn every raster cell centre into its own point.
3. Run some geometric operation (distance, containment, rasterising a polygon) between the geometry and each point.
4. Reshape the flat result back into the `(height, width)` shape of our grid.

Watch the units in the values and plots below. Because we reprojected to a metric CRS earlier, distances come out in metres, not degrees, which is exactly why that reprojection step matters for any calculation that depends on real-world distance.


In [ ]:
from shapely.geometry import Point

# .boundary converts the polygon(s) into their outline (a line), and .union_all()
# merges every part into a single shapely geometry we can measure distances against.
# Doing this once, up front, is much faster than looping over individual boundary segments.
coastline = iceland_m.boundary.union_all()


In [ ]:
coastline


In [ ]:
# Turn every cell centre into its own shapely Point object.
# .ravel() flattens the 2D xx/yy grids to 1D so we can pair up (x, y) coordinates cell by cell.
flat_pts = [Point(x, y) for x, y in zip(xx.ravel(), yy.ravel())]


In [ ]:
flat_pts


In [ ]:
# .distance() gives the straight-line (Euclidean) distance from each point to the coastline,
# in the units of our CRS, metres, since we are working in EPSG:3857.
# reshape() puts the flat list of distances back into the (height, width) grid shape.
d_coast = np.array([p.distance(coastline) for p in flat_pts], dtype="float32").reshape(height, width)


In [ ]:
d_coast


In [ ]:
import geopandas as gpd
import numpy as np
from rasterio.features import rasterize
from shapely.geometry import Point

# rasterize() needs an affine transform, the same kind we built earlier with from_origin(),
# so we rebuild one here directly from the xx/yy coordinate grids.
from affine import Affine
dx = float(np.abs(xx[0,1] - xx[0,0]))     # pixel size in x
dy = float(np.abs(yy[1,0] - yy[0,0]))     # pixel size in y
transform = Affine(dx, 0, xx.min(), 0, -dy, yy.max())

# Burn the Cyprus polygon into the raster grid: cells inside the polygon get 1 (land),
# everything else is left at the fill value, 0 (sea). This is the reverse direction
# from the distance calculation above, here we go from vector polygon to raster grid.
mask = rasterize(
    [(geom, 1) for geom in cyprus.geometry],   # each geometry gets value 1
    out_shape=xx.shape,
    transform=transform,
    fill=0,                                   # sea = 0
    dtype="uint8"
)


In [ ]:
mask


In [ ]:
# Create a normalised north-south gradient (0 = south, 1 = north)
north01 = (yy - yy.min()) / (yy.max() - yy.min())
# Simulate a simple SST field: 26 degrees C in the south, cooling to 23 degrees C in the north
sst = 26.0 - 3.0 * north01

# Mask out land: we only want SST values over the sea.
# mask == 1 inside Cyprus (land), so we keep sst only where mask == 0 (sea) and set land to NaN.
sst = np.where(mask == 0, sst, np.nan)


In [ ]:
sst


In [ ]:
# Plot the synthetic SST raster. origin="upper" matches how we built the grid,
# with row 0 (and yy.max()) at the top, so north stays at the top of the image.
plt.imshow(sst, origin="upper")
plt.title("Synthetic Sea Surface Temperature (°C)")
plt.colorbar(label="SST (°C)")
plt.show()
